In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

In [2]:
baseline_df = pd.read_csv(r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\data\processed\baseline_features.csv")

baseline_df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,air_temp_std_10,process_temp_std_10,rpm_std_10,torque_std_10,tool_wear_std_10,air_temp_var_10,process_temp_var_10,rpm_var_10,torque_var_10,tool_wear_var_10
0,10,M14869,M,298.5,309.0,1741,28.0,21,0,0,...,0.128668,0.134990,113.060652,6.826655,6.834553,0.016556,0.018222,12782.711111,46.603222,46.711111
1,11,H29424,H,298.4,308.9,1782,23.9,24,0,0,...,0.139841,0.152388,140.100004,8.377722,6.988880,0.019556,0.023222,19628.011111,70.186222,48.844444
2,12,H29425,H,298.6,309.1,1423,44.3,29,0,0,...,0.183787,0.200278,138.545460,8.179622,7.734483,0.033778,0.040111,19194.844444,66.906222,59.822222
3,13,M14872,M,298.6,309.1,1339,51.1,34,0,0,...,0.202485,0.213177,153.055582,8.459899,8.769265,0.041000,0.045444,23426.011111,71.569889,76.900000
4,14,M14873,M,298.6,309.2,1742,30.0,37,0,0,...,0.217307,0.236878,162.150684,8.798131,9.569047,0.047222,0.056111,26292.844444,77.407111,91.566667


In [3]:
context_df = pd.read_csv(r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\data\processed\context_fused_dataset.csv")

context_df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,torque_std_10,tool_wear_std_10,air_temp_var_10,process_temp_var_10,rpm_var_10,torque_var_10,tool_wear_var_10,timestamp,ambient_temperature,load_density
0,10,M14869,M,298.5,309.0,1741,28.0,21,0,0,...,6.826655,6.834553,0.016556,0.018222,12782.711111,46.603222,46.711111,2026-01-01 00:00:00,308.618282,0.280
1,11,H29424,H,298.4,308.9,1782,23.9,24,0,0,...,8.377722,6.988880,0.019556,0.023222,19628.011111,70.186222,48.844444,2026-01-01 00:01:00,309.102469,0.239
2,12,H29425,H,298.6,309.1,1423,44.3,29,0,0,...,8.179622,7.734483,0.033778,0.040111,19194.844444,66.906222,59.822222,2026-01-01 00:02:00,308.751006,0.443
3,13,M14872,M,298.6,309.1,1339,51.1,34,0,0,...,8.459899,8.769265,0.041000,0.045444,23426.011111,71.569889,76.900000,2026-01-01 00:03:00,309.768583,0.511
4,14,M14873,M,298.6,309.2,1742,30.0,37,0,0,...,8.798131,9.569047,0.047222,0.056111,26292.844444,77.407111,91.566667,2026-01-01 00:04:00,308.208461,0.300


In [4]:
X_base = baseline_df[["Rotational speed [rpm]","Torque [Nm]","Tool wear [min]"]]

y_base = baseline_df["Machine failure"]

In [5]:
X_base = X_base.select_dtypes(include=np.number)

In [6]:
X_train,X_test,y_train,y_test = train_test_split(X_base,y_base,test_size=0.2,random_state=42)

In [7]:
model_base = RandomForestClassifier(n_estimators=100,max_depth=5,random_state=42)

model_base.fit(X_train,y_train)

RandomForestClassifier(max_depth=5, random_state=42)

In [8]:
base_prediction = model_base.predict(X_test)

In [9]:
base_accuracy = accuracy_score(y_test,base_prediction)

base_f1 = f1_score(y_test,base_prediction)

print("Baseline Accuracy: ", base_accuracy)
print("Baseline F1: ", base_f1)

Baseline Accuracy:  0.967983991995998
Baseline F1:  0.37254901960784315


In [10]:
X_context = context_df.drop("Machine failure",axis=1)

y_context = context_df["Machine failure"]

X_context = X_context.select_dtypes(include=np.number)

In [11]:
X_train2,X_test2,y_train2,y_test2 = train_test_split(X_context,y_context,test_size=0.2,random_state=42)

In [12]:
model_context = RandomForestClassifier(n_estimators=100,max_depth=5,random_state=42)

model_context.fit(X_train2,y_train2)

RandomForestClassifier(max_depth=5, random_state=42)

In [13]:
context_prediction = model_context.predict(X_test2)

In [14]:
context_accuracy = accuracy_score(y_test2,context_prediction)

context_f1 = f1_score(y_test2,context_prediction)

print("Context Accuracy: ", context_accuracy)
print("Context F1: ", context_f1)

Context Accuracy:  0.9979989994997499
Context F1:  0.9736842105263158


In [15]:
comparison = pd.DataFrame({
                           "Model":["Sensor only","Sensor + External Context"], 
                           "Accuracy":[base_accuracy,context_accuracy],
                           "F1 Score":[base_f1,context_f1]
                           })
comparison

,Model,Accuracy,F1 Score
0,Sensor only,0.967984,0.372549
1,Sensor + External Context,0.997999,0.973684


In [16]:
comparison.to_csv(r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\data\processed\ablation_results.csv",index=False)